# Language Modeling using Ngram

In this Exercise, we are going to create a bigram language model and its variation. We will build one model for each of the following type and calculate their perplexity:
- Unigram Model
- Bigram Model
- Bigram Model with Laplace smoothing
- Bigram Model with Interpolation
- Bigram Model with Kneser-ney Interpolation

We will also use NLTK which is a natural language processing library for python to make our lives easier.



In [3]:
# #download corpus
!wget --no-check-certificate https://github.com/ekapolc/nlp_2019/raw/master/HW4/BEST2010.zip
!unzip BEST2010.zip

--2025-02-22 13:15:35--  https://github.com/ekapolc/nlp_2019/raw/master/HW4/BEST2010.zip
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ekapolc/nlp_2019/master/HW4/BEST2010.zip [following]
--2025-02-22 13:15:35--  https://raw.githubusercontent.com/ekapolc/nlp_2019/master/HW4/BEST2010.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7423530 (7.1M) [application/zip]
Saving to: ‘BEST2010.zip’

BEST2010.zip        100%[===================>]   7.08M  --.-KB/s    in 0.09s   

2025-02-22 13:15:36 (77.6 MB/s) - ‘BEST2010.zip’ saved [7423530/7423530]

Archive:  BEST2010.zip
   creating: BEST2010/
  i

In [4]:
!wget https://www.dropbox.com/s/jajdlqnp5h0ywvo/tokenized_wiki_sample.csv

--2025-02-22 13:15:36--  https://www.dropbox.com/s/jajdlqnp5h0ywvo/tokenized_wiki_sample.csv
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/88uzig0mno1b57d6bhwht/tokenized_wiki_sample.csv?rlkey=oya9jw1rljj31jc49fvoaty01 [following]
--2025-02-22 13:15:37--  https://www.dropbox.com/scl/fi/88uzig0mno1b57d6bhwht/tokenized_wiki_sample.csv?rlkey=oya9jw1rljj31jc49fvoaty01
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc89ec800c8c12b9491841c1f9bd.dl.dropboxusercontent.com/cd/0/inline/Ckm2RwSV8QbTuzUzwoh_uPVtUaY7_8uzu2DxZ5KA9yQ8sPOj6feGxCU2lVMUTKQXfHRNPFKvPxsEKck5Ct3cnwdJ6jcbb9Nl0EoLgaPy-ARzTcvg2mjbe63tU5NQs8qmx_STA0cU-Tkzku2RWa_WaWol/file# [following]
--2025-02-22 13:15:37--  https://uc89ec800c8c12b9491841c1f9bd.

In [5]:
#First we import necessary library such as math, nltk, bigram, and collections.
import math
import nltk
import io
import random
from random import shuffle
from nltk import bigrams, trigrams
from collections import Counter, defaultdict
random.seed(999)

BEST2010 is a free Thai NLP dataset by NECTEC usually used as a standard benchmark for various NLP tasks including language modeling. It is separated into 4 domains including article, encyclopedia, news, and novel. The data is already  tokenized using '|' as a separator.

For example,

ตาม|ที่|นางประนอม ทองจันทร์| |กับ| |ด.ช.กิตติพงษ์ แหลมผักแว่น| |และ| |ด.ญ.กาญจนา กรองแก้ว| |ป่วย|สงสัย|ติด|เชื้อ|ไข้|ขณะ|นี้|ยัง|ไม่|ดี|ขึ้น|

In [6]:
total_word_count = 0
best2010 = []
with open('BEST2010/news.txt','r',encoding='utf-8') as f:
  for i,line in enumerate(f):
    line=line.strip()[:-1] #remove the trailing |
    total_word_count += len(line.split("|"))
    best2010.append(line)

In [7]:
#For simplicity, we assumes that each line is a sentence.
print (f'Total sentences in BEST2010 news dataset :\t{len(best2010)}')
print (f'Total word counts in BEST2010 news dataset :\t{total_word_count}')

Total sentences in BEST2010 news dataset :	30969
Total word counts in BEST2010 news dataset :	1660190


We separate the input into 2 sets, train and test data with 70:30 ratio

In [8]:
sentences = best2010
# The data is separated to train and test set with 70:30 ratio.
train = sentences[:int(len(sentences)*0.7)]
test = sentences[int(len(sentences)*0.7):]

#Training data
train_word_count =0
for line in train:
    for word in line.split('|'):
        train_word_count+=1
print ('Total sentences in BEST2010 news training dataset :\t'+ str(len(train)))
print ('Total word counts in BEST2010 news training dataset :\t'+ str(train_word_count))

Total sentences in BEST2010 news training dataset :	21678
Total word counts in BEST2010 news training dataset :	1042797


Here we load the data from Wikipedia which is also already tokenized. It will be used for answering questions in MyCourseville.

In [9]:
import pandas as pd
wiki_data = pd.read_csv("tokenized_wiki_sample.csv")

## Data Preprocessing

Before training any language models, the first step we always do is process the data into the format suited for the LM.

For this exercise, we will use NLTK to help process our data.

In [10]:
from nltk.lm.preprocessing import pad_both_ends, flatten
from nltk.lm.vocabulary import Vocabulary
from nltk import ngrams

We begin by "tokenizing" our training set. Note that the data is already tokenized so we can just split it.

In [11]:
tokenized_train = [["<s>"] + t.split("|") + ["</s>"] for t in train] # "tokenize" and pad each sentence

Next we create a vocabulary with the ```Vocabulary``` class from NLTK. It accepts a list of tokens so we flatten our sentences into one long sentence first.







In [12]:
flat_tokens = list(flatten(tokenized_train)) #join all sentences into one long sentence
vocab = Vocabulary(flat_tokens, unk_cutoff=3) #Words with frequency **below** 3 (not exactly 3) will not be considered in our vocab and will be converted to <UNK>.

Then we replace low frequency words.

Now *each* sentence is going to look something like this:
\["\<s\>", "hello", "my", "name", "is", "\<UNK\>", "\</s\>" \]

In [13]:
tokenized_train = [[token if token in vocab else "<UNK>" for token in sentence] for sentence in tokenized_train]

Finally, we do the same for the test set and the wiki dataset.

In [14]:
tokenized_test = [t.split("|") for t in test]
tokenized_test = [[token if token in vocab else "<UNK>" for token in sentence] for sentence in tokenized_test]

tokenized_wiki_test = [t.split("|") for t in wiki_data['tokenized'].tolist()]
tokenized_wiki_test = [[token if token in vocab else "<UNK>" for token in sentence] for sentence in tokenized_wiki_test]

# Unigram

In this section, we will demonstrate how to build a unigram language model <br>
**Important note:** <br>
**\<s\>** = sentence start symbol <br>
**\</s\>** = sentence end symbol

# VERY IMPORTANT:
- In this notebook, we will *not* default the unknown token probability to ```1/len(vocab)``` but instead will treat it as a normal word and let the model learn its probability so that we can compare our results to NLTK.
- **Also make sure that the code in this notebook can be executed without any problem. If we find that you used NLTK to answer questions in MyCourseVille and did not finish the assignment, you will receive a grade of 0 for this assignment.**

In [15]:
class UnigramModel():
  def __init__(self, data, vocab):
    self.unigram_count = defaultdict(lambda: 0.0)
    self.word_count = 0
    self.vocab = vocab
    for sentence in data:
        for w in sentence: #[(word1, ), (word2, ), (word3, )...]
          w = w[0]
          self.unigram_count[w] +=1.0
          self.word_count+=1

  def __getitem__(self, w):
    w = w[0]  #[(word1, ), (word2, ), (word3, )...]
    if w in self.vocab:
      return self.unigram_count[w]/(self.word_count)
    else:
      return self.unigram_count["<UNK>"]/(self.word_count)

In [16]:
train_unigrams = [list(ngrams(sent, n=1)) for sent in tokenized_train] #creating the unigrams by setting n=1
model = UnigramModel(train_unigrams, vocab)

In [17]:
def getLnValue(x):
      if x == 0:
        return -math.inf
      return math.log(x)

In [18]:
#problability of 'นายก'
print(getLnValue(model[('นายก',)]))

#for example, problability of 'นายกรัฐมนตรี' which is an unknown word is equal to
print(getLnValue(model[('นายกรัฐมนตรี',)]))

#problability of 'นายก' 'ได้' 'ให้' 'สัมภาษณ์' 'กับ' 'สื่อ'
prob = getLnValue(model[('นายก',)])+getLnValue(model[('ได้',)])+ getLnValue(model[('ให้',)])+getLnValue(model[('สัมภาษณ์',)])+getLnValue(model[('กับ',)])+getLnValue(model[('สื่อ',)])+getLnValue(model[('</s>',)])
print ('Problability of a sentence', math.exp(prob))

-6.571687039690381
-3.952132570275872
Problability of a sentence 4.877889285183675e-18


# Perplexity

In order to compare language model we need to calculate perplexity. In this task you should write a perplexity calculation code for the unigram model. The result perplexity should be around 448.90 and
392.74 on train and test data.

## TODO #1 Calculate perplexity

In [19]:
def getLnValue(x):
    if x == 0:
        return -math.inf
    return math.log(x)

def calculate_sentence_ln_prob(sentence, model):
    total_ln_prob = 0.0
    for token in sentence:
        prob = model[token]
        total_ln_prob += getLnValue(prob)
    return total_ln_prob

def perplexity(test, model):
    total_ln_prob = 0.0
    total_tokens = 0
    for sentence in test:
        total_ln_prob += calculate_sentence_ln_prob(sentence, model)
        total_tokens += len(sentence)
    avg_ln_prob = total_ln_prob / total_tokens
    return math.exp(-avg_ln_prob)


In [20]:
test_unigrams = [list(ngrams(sent, n=1)) for sent in tokenized_test]

In [21]:
print(perplexity(train_unigrams,model))
print(perplexity(test_unigrams,model))

448.89690751824827
392.74028966757214


## Q1 MCV
Calculate the perplexity of the model on the wiki test set and answer in MyCourseVille

In [22]:
wiki_test_unigrams = [list(ngrams(sent, n=1)) for sent in tokenized_wiki_test]

In [23]:
print(perplexity([list(flatten(wiki_test_unigrams))], model))

485.7336366066887


# Bigram

Next, you will create a better language model than a unigram (which is not much to compare with). But first, it is very tedious to count every pair of words that occur in our corpus by ourselves. Lucky for us, nltk provides us a simple library which will simplify the process.

In [24]:
#example of nltk usage for bigram
sentence = 'I always search google for an answer .'
padded_sentence = list(pad_both_ends(sentence.split(), n=2))

print('This is how nltk generate bigram.')
for w1,w2 in bigrams(padded_sentence):
    print(w1,w2)
print('\n<s> and </s> are used as a start and end of sentence symbol. respectively.')

This is how nltk generate bigram.
<s> I
I always
always search
search google
google for
for an
an answer
answer .
. </s>

<s> and </s> are used as a start and end of sentence symbol. respectively.


Now, you should be able to implement a bigram model by yourself. Also, you must create a new perplexity calculation for bigram. The result perplexity should be around 56.46 and inf on train and test data.

## TODO #3 Write Bigram Model

In [39]:
class BigramModel():
    def __init__(self, data, vocab):
        # data: list of sentences, where each sentence is a list of bigrams
        self.bigram_counts = defaultdict(float)
        self.unigram_counts = defaultdict(float)
        self.vocab = vocab
        for sentence in data:
            for bigram in sentence:
                w1, w2 = bigram
                self.bigram_counts[(w1, w2)] += 1.0
                self.unigram_counts[w1] += 1.0

    def __getitem__(self, bigram):
        w1, w2 = bigram
        # If w1 has never been seen, or if the bigram count is zero, return probability 0
        if self.unigram_counts[w1] == 0:
            return 0.0
        return self.bigram_counts[bigram] / self.unigram_counts[w1]

## TODO #4 Write Perplexity for Bigram Model

Sum perplexity score at a sentence level, instead of word level

In [40]:
def perplexity(bigram_data, model):
    sentence_perplexities = []
    for sentence in bigram_data:
        if not sentence:  # skip empty sentences
            continue
        sentence_log_prob = 0.0
        for bigram in sentence:
            prob = model[bigram]
            sentence_log_prob += getLnValue(prob)
        # Compute sentence-level perplexity: exp(- average log probability)
        sent_perp = math.exp(-sentence_log_prob / len(sentence))
        sentence_perplexities.append(sent_perp)
    # Return the average perplexity over all sentences.
    return sum(sentence_perplexities) / len(sentence_perplexities) if sentence_perplexities else float('inf')

In [41]:
train_bigrams = [list(ngrams(sent, n=2)) for sent in tokenized_train]
test_bigrams = [list(ngrams(sent, n=2)) for sent in tokenized_test]

In [42]:
bigram_model_scratch = BigramModel(train_bigrams, vocab)

In [43]:
print(perplexity([list(flatten(train_bigrams))], bigram_model_scratch))
print(perplexity([list(flatten(test_bigrams))[:16]], bigram_model_scratch)) #can be used to compare with nltk
print(perplexity([list(flatten(test_bigrams))], bigram_model_scratch))

56.45504870219316
24.598691603876187
inf


## Q2 MCV

In [44]:
wiki_test_bigrams = [list(ngrams(sent, n=2)) for sent in tokenized_wiki_test]

In [45]:
print(perplexity([list(flatten(wiki_test_bigrams))],bigram_model_scratch))

inf


# Smoothing

Usually any ngram models have a sparsity problem, which means it does not have every possible ngram of words in the dataset. Smoothing techniques can alleviate this problem. In this section, you will implement three basic smoothing methods laplace smoothing, interpolation for bigram, and Knesey-Ney smoothing.

## TODO #5 write Bigram with Laplace smoothing (Add-One Smoothing)

The result perplexity on training and testing should be:

    370.28, 369.16 for Laplace smoothing

In [46]:
class BigramWithLaplaceSmoothing():
    def __init__(self, data, vocab):
        self.bigram_counts = defaultdict(float)
        self.unigram_counts = defaultdict(float)
        self.vocab = vocab
        self.vocab_size = len(vocab)  # total number of words in vocabulary
        for sentence in data:
            for bigram in sentence:
                w1, w2 = bigram
                self.bigram_counts[(w1, w2)] += 1.0
                self.unigram_counts[w1] += 1.0

    def __getitem__(self, bigram):
        w1, w2 = bigram
        # Add-one smoothing: add one to numerator and vocab_size to denominator.
        numerator = self.bigram_counts[(w1, w2)] + 1.0
        denominator = self.unigram_counts[w1] + self.vocab_size
        return numerator / denominator

# Build the model and test perplexity.
model = BigramWithLaplaceSmoothing(train_bigrams, vocab)
print(perplexity([list(flatten(train_bigrams))], model))
print(perplexity([list(flatten(test_bigrams))], model))

370.28232024056035
369.1605485652555


## Q3 MCV

In [47]:
print(perplexity([list(flatten(wiki_test_bigrams))],model))

735.0253999215859


## TODO #6 Write Bigram with Interpolation
Set the lambda value as 0.7 for bigram, 0.25 for unigram, and 0.05 for unknown word.

The result perplexity on training and testing should be:

    70.07, 102.99 for Interpolation

In [52]:
class BigramWithInterpolation():
    def __init__(self, data, vocab):
        self.vocab = vocab
        self.bigram_counts = defaultdict(float)
        self.unigram_counts = defaultdict(float)
        for sentence in data:
            for bigram in sentence:
                w1, w2 = bigram
                self.bigram_counts[(w1, w2)] += 1.0
                self.unigram_counts[w1] += 1.0
            # Also count the final token in the sentence
            if sentence:
                self.unigram_counts[sentence[-1][1]] += 1.0
        self.total_words = sum(self.unigram_counts.values())
        self.vocab_size = len(vocab)

    def __getitem__(self, bigram):
        w1, w2 = bigram
        # Bigram probability
        if self.unigram_counts[w1] > 0:
            p_bigram = self.bigram_counts[(w1, w2)] / self.unigram_counts[w1]
        else:
            p_bigram = 0.0
        # Unigram probability computed over the entire training corpus
        p_unigram = self.unigram_counts[w2] / self.total_words if self.total_words > 0 else 0.0
        # For interpolation, we set the unknown probability as a uniform probability over the vocabulary.
        p_unknown = 1.0 / self.vocab_size
        # Interpolated probability with the specified lambda values
        return 0.7 * p_bigram + 0.25 * p_unigram + 0.05 * p_unknown

# Example usage:
model = BigramWithInterpolation(train_bigrams, vocab)
print(perplexity([list(flatten(train_bigrams))], model))
print(perplexity([list(flatten(test_bigrams))], model))

70.06754793707988
102.99295310997627


## Q4 MCV

In [53]:
print(perplexity([list(flatten(wiki_test_bigrams))],model))

251.60709095654997


## Language modeling on multiple domains

Sometimes, we do not have enough data to create a language model for a new domain. In that case, we can improvised by combining several models to improve result on the new domain.

In this exercise you will try to merge two language models from news and article domains to create a language model for the encyclopedia domain.

In [36]:
# create encyclopeida data (test data)
encyclo_data=[]
with open('BEST2010/encyclopedia.txt','r',encoding='utf-8') as f:
    for i,line in enumerate(f):
        encyclo_data.append(line.strip()[:-1])

(news) First, you should try to calculate perplexity of your bigram with interpolation on encyclopedia data. The  perplexity should be around 236.33

In [54]:
# Tokenize encyclopedia data and add start and end symbols
tokenized_encyclopedia = [["<s>"] + line.split("|") + ["</s>"] for line in encyclo_data]

# Replace words not in the vocabulary with <UNK>
tokenized_encyclopedia = [
    [token if token in vocab else "<UNK>" for token in sent]
    for sent in tokenized_encyclopedia
]

# Create bigrams for each encyclopedia sentence
encyclopedia_bigrams = [list(ngrams(sent, 2)) for sent in tokenized_encyclopedia]

In [55]:
# 1) news only on "encyclopedia"
print(perplexity([list(flatten(encyclopedia_bigrams))], model))

236.3286142031601


## TODO #7 - Langauge Modelling on Multiple Domains
Combine news and article datasets to create another bigram model and evaluate it on the encyclopedia data.



(article) For your information, a bigram model with interpolation using article data to test on encyclopedia data has a perplexity of 218.57

In [63]:
# 2) article only on "encyclopedia"
best2010_article=[]
with open('BEST2010/article.txt','r',encoding='utf-8') as f:
    for i,line in enumerate(f):
        best2010_article.append(line.strip()[:-1])

combined_total_word_count = 0
for line in best2010_article:
    combined_total_word_count += len(line.split('|'))

# Tokenize article data and pad with start and end symbols
tokenized_article = [["<s>"] + line.split("|") + ["</s>"] for line in best2010_article]

# Build article vocabulary from the flattened tokens; words with frequency below 3 are mapped to <UNK>
from nltk.lm.vocabulary import Vocabulary
from nltk.lm.preprocessing import flatten
flat_article_tokens = list(flatten(tokenized_article))
article_vocab = Vocabulary(flat_article_tokens, unk_cutoff=3)

# Use the article vocabulary when processing encyclopedia data:
tokenized_encyclopedia_article = [
    ["<s>"] + line.split("|") + ["</s>"] for line in encyclo_data
]
tokenized_encyclopedia_article = [
    [token if token in article_vocab else "<UNK>" for token in sent]
    for sent in tokenized_encyclopedia_article
]
encyclopedia_bigrams = [list(ngrams(sent, 2)) for sent in tokenized_encyclopedia_article]


model = BigramWithInterpolation(article_bigrams, article_vocab)

In [64]:
print('Perplexity of the bigram model using article data with interpolation smoothing on encyclopedia test data',perplexity([list(flatten(encyclopedia_bigrams))], model))

Perplexity of the bigram model using article data with interpolation smoothing on encyclopedia test data 218.57479345888848


In [68]:
# 3) train on news + article, test on "encyclopedia"
best2010_article_and_news = best2010_article.copy()
with open('BEST2010/news.txt','r',encoding='utf-8') as f:
    for i,line in enumerate(f):
        best2010_article_and_news.append(line.strip()[:-1])

# Tokenize the combined data and pad with <s> and </s>
tokenized_combined = [["<s>"] + line.split("|") + ["</s>"] for line in best2010_article_and_news]

# Build a vocabulary from the combined data
flat_combined_tokens = list(flatten(tokenized_combined))
combined_vocab = Vocabulary(flat_combined_tokens, unk_cutoff=3)

# แทนคำที่มีความถี่ต่ำด้วย <UNK>
tokenized_combined = [
    [token if token in combined_vocab else "<UNK>" for token in sent]
    for sent in tokenized_combined
]

# สร้าง bigrams สำหรับข้อมูล combined
from nltk import ngrams
combined_bigrams = [list(ngrams(sent, 2)) for sent in tokenized_combined]

combined_model = BigramWithInterpolation(combined_bigrams, combined_vocab)

# Re-tokenize ข้อมูล encyclopedia โดยใช้ combined_vocab
tokenized_encyclopedia = [["<s>"] + line.split("|") + ["</s>"] for line in encyclo_data]
tokenized_encyclopedia = [
    [token if token in combined_vocab else "<UNK>" for token in sent]
    for sent in tokenized_encyclopedia
]
encyclopedia_bigrams = [list(ngrams(sent, 2)) for sent in tokenized_encyclopedia]

print('Perplexity of the combined Bigram model with interpolation smoothing on encyclopedia test data',perplexity([list(flatten(encyclopedia_bigrams))], combined_model))

Perplexity of the combined Bigram model with interpolation smoothing on encyclopedia test data 242.88025282580364


## TODO #8 - Kneser-ney on "News"

<!-- Reimplement equation 4.33 in SLP textbook (https://lagunita.stanford.edu/c4x/Engineering/CS-224N/asset/slp4.pdf) -->

Implement Bigram Knerser-ney LM. The result perplexity should be around 65.81, 93.21 on train and test data. Be careful not to mix up vocab from the above section!


In [69]:
class BigramKneserNey():
    def __init__(self, data, vocab):
        self.vocab = vocab
        self.discount = 0.75  # ค่า discount ที่นิยมใช้
        self.bigram_counts = defaultdict(float)
        self.unigram_counts = defaultdict(float)  # นับจำนวนการเกิดของ w1 ใน bigram
        self.following = defaultdict(set)         # สำหรับแต่ละ w1, เก็บชุดของ w2 ที่ตามหลัง
        self.preceding = defaultdict(set)         # สำหรับแต่ละ w2, เก็บชุดของ w1 ที่ปรากฏก่อนหน้า

        # นับจำนวน bigram และเก็บ context ต่างๆ
        for sentence in data:
            for bigram in sentence:
                w1, w2 = bigram
                self.bigram_counts[(w1, w2)] += 1.0
                self.unigram_counts[w1] += 1.0
                self.following[w1].add(w2)
                self.preceding[w2].add(w1)

        # จำนวน unique bigram types สำหรับ P_continuation
        self.total_bigram_types = len(self.bigram_counts)

    def __getitem__(self, bigram):
        w1, w2 = bigram
        # Count ของ bigram (w1, w2)
        c = self.bigram_counts[(w1, w2)]
        # Count ของ w1 ในตำแหน่งแรกของ bigram
        c_w1 = self.unigram_counts[w1]
        # จำนวน unique continuations ที่ตามหลัง w1
        unique_continuations = len(self.following[w1]) if c_w1 > 0 else 0
        # P_continuation(w2) = (จำนวน unique w1 ที่ปรากฏก่อน w2) / (จำนวน unique bigram ทั้งหมด)
        cont_count = len(self.preceding[w2])
        p_cont = cont_count / self.total_bigram_types if self.total_bigram_types > 0 else 0.0

        # คำนวณ probability แบบ Kneser-Ney:
        # P(w2|w1) = max(c(w1,w2) - d, 0) / c(w1) + (d * (number of unique continuations for w1) / c(w1)) * P_continuation(w2)
        p = (max(c - self.discount, 0) / c_w1) if c_w1 > 0 else 0.0
        p += ((self.discount * unique_continuations) / c_w1 if c_w1 > 0 else 0.0) * p_cont
        return p

# สร้าง model ด้วย train_bigrams และ vocab ที่กำหนดไว้
model = BigramKneserNey(train_bigrams, vocab)

# คำนวณ perplexity ด้วยฟังก์ชัน perplexity ที่ใช้ร่วมกันอยู่แล้ว
print(perplexity([list(flatten(train_bigrams))], model))
print(perplexity([list(flatten(train_bigrams))[:1000]], model))  # เปรียบเทียบกับ nltk
print(perplexity([list(flatten(test_bigrams))[:1000]], model))   # เปรียบเทียบกับ nltk
print(perplexity([list(flatten(test_bigrams))], model))


65.80901349887752
50.40541747450013
90.29909158711666
93.21479572741401


## Q5 MCV

In [70]:
print(perplexity([list(flatten(wiki_test_bigrams))],model))

266.582022413098
